In [1]:
import logging
from pyspark import SparkConf
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import month, days
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType, TimestampNTZType

In [2]:
from pyspark.sql import SparkSession


def create_spark_session(app_name: str) -> SparkSession:
    spark = (
        SparkSession.builder
        .master("spark://spark-master:7077")
        .appName(app_name)
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.catalog.lakehouse_prod","org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.lakehouse_prod.type", "hive")
        .config("spark.sql.catalog.lakehouse_prod.uri","thrift://hive-metastore:9083")
        .config("spark.sql.catalog.lakehouse_prod.warehouse","s3a://lakehouse-prod-bucket/warehouse/")
        .config("spark.sql.catalog.lakehouse_prod.io-impl","org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .enableHiveSupport()
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")
    print("NOTE: SparkSession created successfully!")

    return spark

In [3]:
app_name = 'Spark-Iceberg-Test'
spark = create_spark_session(app_name)
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/23 15:39:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


NOTE: SparkSession created successfully!


In [4]:
sc = spark.sparkContext
sc

<SparkContext master=spark://spark-master:7077 appName=Spark-Iceberg-Test>

In [ ]:
spark.sql("SHOW DATABASES IN lakehouse_prod").show()

In [ ]:
spark.sql("USE lakehouse_prod").show()

In [ ]:
spark.sql("SHOW CATALOGS;").show()

In [ ]:
spark.sql("SHOW NAMESPACES").show()

In [ ]:
spark.sql("SHOW TABLES in nyc;").show()

In [ ]:
# spark.sql("""
# DROP DATABASE nyc
# """).show()

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse_prod.nyc").show()

In [ ]:
spark.sql(
"""
CREATE TABLE IF NOT EXISTS lakehouse_prod.nyc.taxis
(
  trip_id bigint,
  trip_distance float,
  fare_amount double,
  store_and_fwd_flag string
)
USING iceberg
LOCATION 's3a://lakehouse-dev-bucket/test_iceberg/nyc_taxis';

"""
).show()

In [ ]:
spark.sql("""
DESCRIBE EXTENDED lakehouse_prod.nyc.taxis
""").show(truncate=False)

In [ ]:
spark.sql("SHOW CATALOGS").show()

In [4]:
# Define schema for NYC Taxi Data
schema = StructType([
    StructField('VendorID', LongType(), True),
    StructField('tpep_pickup_datetime', TimestampNTZType(), True),
    StructField('tpep_dropoff_datetime', TimestampNTZType(), True),
    StructField('passenger_count', DoubleType(), True),
    StructField('trip_distance', DoubleType(), True),
    StructField('RatecodeID', DoubleType(), True),
    StructField('store_and_fwd_flag', StringType(), True),
    StructField('PULocationID', LongType(), True),
    StructField('DOLocationID', LongType(), True),
    StructField('payment_type', LongType(), True),
    StructField('fare_amount', DoubleType(), True),
    StructField('extra', DoubleType(), True),
    StructField('mta_tax', DoubleType(), True),
    StructField('tip_amount', DoubleType(), True),
    StructField('tolls_amount', DoubleType(), True),
    StructField('improvement_surcharge', DoubleType(), True),
    StructField('total_amount', DoubleType(), True),
    StructField('congestion_surcharge', DoubleType(), True),
    StructField('airport_fee', DoubleType(), True)
])

In [5]:
# Read Parquet file from MinIO
parquet_path = './data/yellow_tripdata_2022-04.parquet'
df = spark.read.option("header", "true").schema(schema).parquet(parquet_path)
df.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2022-04-01 00:21:13|  2022-04-01 00:58:33|            1.0|         10.3|       1.0|                 N|         163|          62|           1|       33.5|  3.0|    0.5|      7.4

In [ ]:
# df.write.mode("overwrite").saveAsTable("nyc.yellow_trip_data_202204")

In [6]:
df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [ ]:
# df.writeTo("lakehouse_prod.nyc.taxis_01") \
#   .option("location", "s3a://lakehouse-dev-bucket/external_test/taxis_01") \
#   .partitionedBy(days("tpep_pickup_datetime")) \
#   .replace()

In [ ]:
df.writeTo("lakehouse_prod.nyc.taxis_01") \
  .option("location", "s3a://lakehouse-dev-bucket/external_test/taxis_01") \
  .partitionedBy(days("tpep_pickup_datetime")) \
  .create()

In [ ]:
spark.sql("""
    SELECT * FROM lakehouse_prod.nyc.taxis_01
""").show(5)

In [9]:
df.writeTo("lakehouse_prod.nyc.taxis_02") \
    .tableProperty("location", "s3a://lakehouse-dev-bucket/external_test/taxis_02") \
    .createOrReplace()

In [10]:
df.writeTo("lakehouse_prod.nyc.taxis_02") \
    .overwritePartitions()

In [11]:
df = spark.read.format("iceberg").load("lakehouse_prod.nyc.taxis_02")
df.show()

[Stage 4:>                                                          (0 + 1) / 1]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2022-04-01 00:21:13|  2022-04-01 00:58:33|            1.0|         10.3|       1.0|                 N|         163|          62|           1|       33.5|  3.0|    0.5|      7.4

In [ ]:
spark.stop()